# Find small / far objects in the Egyptian images (Colab)

Runs the CHV-trained YOLOv8m on the 133 Egyptian images **in tiles** (SAHI), so small far-away
people, helmets and vests are found too. Output: `egy_tiled_candidates.csv` (small file) to send back.
These are only **candidates**; every new box is checked by eye before it goes into the labels.

**Before running:** finish the training notebook (experiment A). This notebook uses its
`repro_yolov8m_chv/weights/best.pt` from Google Drive.

Runtime → T4 GPU → Run all. About 5–10 minutes.

In [ ]:
!pip -q install ultralytics sahi gdown
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
WEIGHTS = "/content/drive/MyDrive/ppe_project/runs/repro_yolov8m_chv/weights/best.pt"
EGY_DRIVE_ID = "1Y30bp2I8c5RGSxtRb0Xa6t5RjGmzxx8n"
CONF = 0.15          # low on purpose: we want every possible candidate, false ones get rejected later
SLICE = 640
OVERLAP = 0.25
KEEP = ["IMG (1).JPG", "IMG (3).JPG", "IMG (5).JPG", "IMG (10).JPG", "IMG (12).JPG", "IMG (13).JPG", "IMG (14).JPG", "IMG (15).JPG", "IMG (17).JPG", "IMG (19).JPG", "IMG (20).JPG", "IMG (22).JPG", "IMG (23).JPG", "IMG (24).JPG", "IMG (25).JPG", "IMG (26).JPG", "IMG (27).JPG", "IMG (28).JPG", "IMG (29).JPG", "IMG (30).JPG", "IMG (32).JPG", "IMG (36).JPG", "IMG (37).JPG", "IMG (40).JPG", "IMG (41).JPG", "IMG (42).JPG", "IMG (45).JPG", "IMG (47).JPG", "IMG (49).JPG", "IMG (51).JPG", "IMG (52).JPG", "IMG (53).JPG", "IMG (54).JPG", "IMG (55).JPG", "IMG (56).JPG", "IMG (58).JPG", "IMG (59).JPG", "IMG (60).JPG", "IMG (61).JPG", "IMG (62).JPG", "IMG (63).JPG", "IMG (64).JPG", "IMG (65).JPG", "IMG (66).JPG", "IMG (68).JPG", "IMG (69).JPG", "IMG (72).JPG", "IMG (73).JPG", "IMG (77).JPG", "IMG (79).JPG", "IMG (83).JPG", "IMG (85).JPG", "IMG (86).JPG", "IMG (87).JPG", "IMG (88).JPG", "IMG (93).JPG", "IMG (94).JPG", "IMG (95).JPG", "IMG (97).JPG", "IMG (99).JPG", "IMG (100).JPG", "IMG (101).JPG", "IMG (102).JPG", "IMG (103).JPG", "IMG (106).JPG", "IMG (107).JPG", "IMG (109).JPG", "IMG (111).JPG", "IMG (112).JPG", "IMG (113).JPG", "IMG (116).JPG", "IMG (118).JPG", "IMG (121).JPG", "IMG (123).JPG", "IMG (124).JPG", "IMG (126).JPG", "IMG (127).JPG", "IMG (128).JPG", "IMG (129).JPG", "IMG (132).JPG", "IMG (133).JPG", "IMG (134).JPG", "IMG (135).JPG", "IMG (136).JPG", "IMG (137).JPG", "IMG (138).JPG", "IMG (139).JPG", "IMG (140).JPG", "IMG (142).JPG", "IMG (143).JPG", "IMG (144).JPG", "IMG (146).JPG", "IMG (148).JPG", "IMG (150).JPG", "IMG (152).JPG", "IMG (155).JPG", "IMG (156).JPG", "IMG (157).JPG", "IMG (158).JPG", "IMG (159).JPG", "IMG (162).JPG", "IMG (164).JPG", "IMG (166).JPG", "IMG (168).JPG", "IMG (170).JPG", "IMG (172).JPG", "IMG (7).JPG", "IMG (9).JPG", "IMG (31).JPG", "IMG (33).JPG", "IMG (35).JPG", "IMG (67).JPG", "IMG (84).JPG", "IMG (92).JPG", "IMG (98).JPG", "IMG (104).JPG", "IMG (114).JPG", "IMG (147).JPG", "IMG (154).JPG", "IMG (11).JPG", "IMG (16).JPG", "IMG (18).JPG", "IMG (34).JPG", "IMG (44).JPG", "IMG (71).JPG", "IMG (80).JPG", "IMG (89).JPG", "IMG (91).JPG", "IMG (96).JPG", "IMG (108).JPG", "IMG (120).JPG", "IMG (145).JPG", "IMG (160).JPG"]            # the 133 images left after removing near-duplicates (D-015)

In [ ]:
import os, subprocess
from pathlib import Path
assert Path(WEIGHTS).exists(), "best.pt not found - run the training notebook first, or fix WEIGHTS"
W = Path("/content/egy"); W.mkdir(exist_ok=True); os.chdir(W)
if not Path("egy.rar").exists():
    subprocess.run(["gdown", "--fuzzy", f"https://drive.google.com/file/d/{EGY_DRIVE_ID}/view", "-O", "egy.rar"], check=True)
subprocess.run("apt-get -qq install -y unrar > /dev/null", shell=True, check=True)
subprocess.run(["unrar", "x", "-o+", "-idq", "egy.rar", "raw/"], check=True)
raw = {p.name: p for p in Path("raw").rglob("*") if p.suffix.lower() in (".jpg", ".jpeg", ".png")}
missing = [k for k in KEEP if k not in raw]
print("found", len(raw), "images; missing from KEEP:", missing)

In [ ]:
import csv, numpy as np
from PIL import Image, ImageOps
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction

det = AutoDetectionModel.from_pretrained(model_type="ultralytics", model_path=WEIGHTS,
                                         confidence_threshold=CONF, device="cuda:0")
rows = []
for i, name in enumerate(KEEP):
    im = ImageOps.exif_transpose(Image.open(raw[name])).convert("RGB")   # same orientation as our local copies
    Wd, Ht = im.size
    res = get_sliced_prediction(np.array(im), det, slice_height=SLICE, slice_width=SLICE,
                                overlap_height_ratio=OVERLAP, overlap_width_ratio=OVERLAP,
                                perform_standard_pred=True, postprocess_type="NMS",
                                postprocess_match_metric="IOU", postprocess_match_threshold=0.5,
                                verbose=0)
    for o in res.object_prediction_list:
        b = o.bbox
        rows.append([Path(name).stem, o.category.id, round(o.score.value, 3),
                     round((b.minx + b.maxx) / 2 / Wd, 6), round((b.miny + b.maxy) / 2 / Ht, 6),
                     round((b.maxx - b.minx) / Wd, 6), round((b.maxy - b.miny) / Ht, 6)])
    if i % 20 == 0:
        print(i, name, len(rows))

with open("/content/egy_tiled_candidates.csv", "w", newline="") as f:
    csv.writer(f).writerows([["image", "class_id", "conf", "x", "y", "w", "h"], *rows])
print("boxes:", len(rows))
from google.colab import files
files.download("/content/egy_tiled_candidates.csv")